# Copycat

Copycat is a small concatenative language in which a model invocation is a
first-class program-synthesis effect. The deterministic runtime stays in
control: `{natural language}` asks a backend for Copycat code, and that code
continues immediately against the current data stack.

The language implementation lives in the installable `copycat` package. This
notebook contains its interactive tests and examples.


## Setup — run this section first

Run these cells in Google Colab to install Copycat from GitHub and import its
public API. The optional Gemma dependencies are installed only in the live-model
section below.


### Install the package


In [ ]:
%pip install -q "copycat-language[test] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
from copycat import (
    CopycatError,
    EvaluationError,
    GeneratedCodeError,
    Gemma4Backend,
    Model,
    ModelProtocolError,
    ModelReportedError,
    ParseError,
    StubModel,
    load_module,
    read,
    run,
    save_module,
)


## Tests


In [ ]:
import ipytest
import pytest

ipytest.autoconfig()


In [ ]:
%%ipytest -q


@pytest.mark.parametrize(
    "source, expected",
    [
        ("[foo] d", "[foo] [foo]"),
        ("[foo] e", ""),
        ("[foo] [bar] f", "[bar] [foo]"),
        ("[foo] [bar] c", "[foo bar]"),
        ("[foo] b", "[[foo]]"),
        ("[foo] a", "foo"),
        ("[foo] s bar qux r baz", "[bar qux] foo baz"),
    ],
)
def test_original_examples(source, expected):
    assert run(source, verbose=False) == expected


def test_shift_finds_reset_when_reset_is_the_final_instruction():
    assert run("[a] s 1 r", verbose=False) == "1"


def test_model_form_is_opaque_to_copycat_syntax():
    program = read('{write [this] and "that"\non two lines}')
    (model,) = program.body
    assert isinstance(model, Model)
    assert model.prompt == 'write [this] and "that"\non two lines'


@pytest.mark.parametrize(
    "source, fragment",
    [
        ("[d", "Unclosed quotation"),
        ("d]", "no matching '['"),
        ("{do something", "Unclosed model invocation"),
        ('"unterminated', "unterminated string"),
        ("d @", "Unexpected character '@'"),
        ("g", "Single-letter names are reserved"),
        ("Foo", "Unexpected character 'F'"),
        ("foo_bar", "Unexpected character '_'"),
    ],
)
def test_parser_errors_are_explanatory(source, fragment):
    with pytest.raises(ParseError) as caught:
        read(source)
    assert fragment.lower() in str(caught.value).lower()


def test_hyphenated_user_word_names_parse():
    assert str(read("foo-doc long-word-2")) == "foo-doc long-word-2"


def test_strict_evaluation_reports_stack_underflow():
    with pytest.raises(EvaluationError) as caught:
        run("d", strict=True, verbose=False)

    message = str(caught.value)
    assert "d needs 1 value" in message
    assert "line 1, column 1" in message


def test_dictionary_bodies_are_source_text():
    dictionary = {
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value."',
    }

    assert run("1 duplicate", dictionary=dictionary, verbose=False) == "1 1"
    assert run("duplicate-doc", dictionary=dictionary, verbose=False) == (
        '"Duplicate the top value."'
    )


def test_dictionary_rejects_single_letter_user_words():
    with pytest.raises(ValueError, match="longer than one character"):
        run("1", dictionary={"g": "d"}, verbose=False)


def test_module_round_trip(tmp_path):
    dictionary = {
        "duplicate": "d",
        "duplicate-doc": '"Duplicate the top value."',
        "duplicate-twice": "d d",
    }
    path = tmp_path / "example.module"

    save_module(dictionary, path)

    assert load_module(path) == dictionary


def test_stub_model_ok_executes_generated_code_immediately():
    backend = StubModel("<OK>f</OK>")

    assert run(
        "1 2 {swap the top two values}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "2 1"

    assert backend.calls == [
        ("swap the top two values", "1 2")
    ]


def test_stub_model_error_becomes_structured_condition():
    backend = StubModel("<ERROR>I cannot do that safely.</ERROR>")

    with pytest.raises(ModelReportedError):
        run(
            "1 {do something impossible}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_protocol_allows_surrounding_text_and_lowercase_tags():
    backend = StubModel("I chose this.\n<ok>d</ok>\nDone.")

    assert run(
        "1 {duplicate the value}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "1 1"


def test_protocol_rejects_multiple_expected_elements():
    backend = StubModel("<OK>d</OK> or <ERROR>unsure</ERROR>")

    with pytest.raises(ModelProtocolError):
        run(
            "1 {duplicate the value}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_bad_generated_syntax_is_attributed_to_model_output():
    backend = StubModel("<OK>[d</OK>")

    with pytest.raises(GeneratedCodeError) as caught:
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )

    assert "<model output>" in str(caught.value)


def test_nested_model_calls_are_disabled_by_default():
    backend = StubModel("<OK>{ask again}</OK>")

    with pytest.raises(GeneratedCodeError):
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


## Examples


Run this section in Colab to exercise the reader, dictionaries, deterministic
model effects, then load Gemma and run the live examples. Evaluator tracing is
intentionally enabled in the executable examples.


### Reader examples


In [ ]:
for source in [
    "1 2 f",
    "[foo] s bar qux r baz",
    "foo-doc",
    '{write [this] and "that" on two lines}',
]:
    print(f"{source!r} -> {read(source)!r}")


### Dictionary and module examples


In [ ]:
dictionary = {
    "duplicate": "d",
    "duplicate-doc": '"Duplicate the top value."',
}

print(run("1 duplicate", dictionary=dictionary))
print(run("duplicate-doc", dictionary=dictionary))

save_module(dictionary, "example.module")
loaded_dictionary = load_module("example.module")
print(loaded_dictionary)


### Deterministic model examples


In [ ]:
swap_stub = StubModel("Reasoning outside the tag is accepted.\n<ok>f</ok>")
copy_stub = StubModel("<OK>d</OK>")

print(
    run(
        "1 2 {put the top two values in the opposite order}",
        model_backend=swap_stub,
        strict=True,
    )
)

print(
    run(
        '"hello" {duplicate the top value}',
        model_backend=copy_stub,
        strict=True,
    )
)


### Live Gemma 4 synthesis


#### Load Gemma


In [ ]:
%pip install -q -U "copycat-language[gemma] @ git+https://github.com/dearmadisonblue/copycat.git@main"


In [ ]:
# Optional, only if your Hugging Face environment asks for authentication:
# from huggingface_hub import notebook_login
# notebook_login()

gemma = Gemma4Backend.load(
    max_new_tokens=8_192,
    stream_output=True,
)


#### Run live examples


In [ ]:
examples = [
    "1 2 {put the top two values in the opposite order}",
    '"hello" {duplicate the top value}',
    "{put the number 7 on the stack}",
]

for source in examples:
    print("\n" + "=" * 72)
    print("SOURCE:", source)
    try:
        print(
            "RESULT:",
            run(
                source,
                model_backend=gemma,
                strict=True,
            ),
        )
    except CopycatError as exc:
        print(exc)


## Future work

This version remains deliberately narrow: one model turn synthesizes a small
Copycat program, that program is parsed, and ordinary evaluation continues.
Dictionary reflection is deliberately deferred until the effect and handler
model is settled. Repair loops, generic effect handlers, capabilities, external
services, simulation, persisted continuations, actors, and nested model effects
also remain future work.
